In [1]:
import sys
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import load_yaml_config
from src.data import download_data, load_data_from_json

config = load_yaml_config(project_root / "config.yaml")
city = "Lisboa"

In [2]:
# Download data if it isn't already downloaded
download_data(city, config)

HTTP status: 200
Request URL: https://archive-api.open-meteo.com/v1/archive?latitude=38.7223&longitude=-9.1393&start_date=2020-01-01&end_date=2025-12-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Cwind_speed_10m&timezone=Europe%2FLisbon


In [3]:
# Load data into a DataFrame
df = load_data_from_json(city, config)

In [4]:
display(df)

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2020-01-01 00:00:00,8.6,95,0.0,6.7
1,2020-01-01 01:00:00,10.0,92,0.0,7.7
2,2020-01-01 02:00:00,9.2,94,0.0,7.4
3,2020-01-01 03:00:00,9.2,94,0.0,8.1
4,2020-01-01 04:00:00,9.0,95,0.0,8.9
...,...,...,...,...,...
52603,2025-12-31 19:00:00,10.4,86,0.0,6.1
52604,2025-12-31 20:00:00,10.4,87,0.0,5.7
52605,2025-12-31 21:00:00,10.6,87,0.0,5.5
52606,2025-12-31 22:00:00,10.5,89,0.0,9.0


In [5]:
# Get the information of all data types present in the DataFrame
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52608 entries, 0 to 52607
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  52608 non-null  datetime64[us]
 1   temperature_2m        52608 non-null  float64       
 2   relative_humidity_2m  52608 non-null  int64         
 3   precipitation         52608 non-null  float64       
 4   wind_speed_10m        52608 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1)
memory usage: 2.0 MB


In [6]:
# Check the percentage of missing values for each column:
round(df.isna().sum() / len(df) * 100, 2)

time                    0.0
temperature_2m          0.0
relative_humidity_2m    0.0
precipitation           0.0
wind_speed_10m          0.0
dtype: float64

In [7]:
# Check basic duplicated data
df.duplicated().sum()

np.int64(0)

In [8]:
# Create the rain event
df["rain_event"] = (df["precipitation"] >= 0.1).astype(int)

In [9]:
df["rain_event"].value_counts()

rain_event
0    46260
1     6348
Name: count, dtype: int64

In [10]:
# Check percentage of expected rain event
df["rain_event"].value_counts(normalize=True)

rain_event
0    0.879334
1    0.120666
Name: proportion, dtype: float64

In [ ]:
# Sort by time and collect precipitation from the next three hours.
df = df.sort_values("time").reset_index(drop=True)

future_precip = pd.concat(
    [df["precipitation"].shift(-hour) for hour in (1,2,3)],
    axis=1
)

In [ ]:
# Mark whether it rains at any point in the next three hours.
df["rain_next_3h"] = future_precip.ge(0.1).any(axis=1).astype(int)
df.loc[future_precip.isna().any(axis=1), "rain_next_3h"] = pd.NA

In [ ]:
# Check the counts of rainy, dry, and unlabeled hours.
df["rain_next_3h"].value_counts(dropna=False)

rain_next_3h
0.0    43293
1.0     9312
NaN        3
Name: count, dtype: int64